In [ ]:
polaris_object_dir = '/Users/kense/Library/CloudStorage/OneDrive-Personal/work/hog/polaris_result/a/57/resulthi90'

object_id = '810'
polaris_object_dir = '/Volumes/MWISP/work/polaris_object/'+str(object_id)
hdr_file = '/Users/kense/Library/CloudStorage/OneDrive-Personal/work/polaris_hdr.fits'

In [ ]:
import os
import glob
from astropy.io import fits
import numpy as np
from matplotlib import pyplot as plt

# Set the directory to search.
folder_path = polaris_object_dir+'/resulthi90/data'

# Set the output directory for the new FITS files.
output_folder_path = polaris_object_dir

# Find all matching FITS files in the directory.
fits_files = sorted(glob.glob(os.path.join(folder_path, 'vel_channel_maps_species_0001_line_0001_vel_*.fits')))
print(len(fits_files))

# Create the FITS header.
merge_header = fits.open(hdr_file)[0].header
polaris_object_header = fits.open(fits_files[0])[0].header

merge_header['CDELT1'] = polaris_object_header['CDELT1A']
merge_header['CDELT2'] = polaris_object_header['CDELT2A']

v_range = 50
n_channel = len(fits_files)
dv = v_range*1000 / n_channel

merge_header['CDELT3'] = dv



# Initialize lists to collect each data[i] component across files.
data_lists = [[] for _ in range(5)]  # Use components data[0] through data[4].

# Loop over all FITS files.
for file_path in fits_files:
    with fits.open(file_path) as hdul:
        data = hdul[0].data
        # Process each data[i] component.
        for i in range(5):
            data_lists[i].append(data[i])

merged_data = []


# Combine each component across files and save it as a new FITS file.
for i, data_list in enumerate(data_lists):
    # Stack the arrays along axis 0.
    merged_data.append( np.stack(data_list, axis=0) )
    
    # Create a new FITS file.
    hdu = fits.PrimaryHDU(merged_data[0][:,50:250,50:250],header=merge_header) 
    
    # Set the new FITS filename.
    new_file_path = os.path.join(output_folder_path, f'Cube_'+str(object_id)+f'_{i}.fits')
    
    # Save the new FITS file.
    hdu.writeto(new_file_path, overwrite=True)
    merged_data = []




In [ ]:
cube0 = fits.open(os.path.join(output_folder_path, 'Cube_'+str(object_id)+'_0.fits'))[0]
# Initialize a zero-filled array with the same shape as the original data.
rotated_data = np.zeros_like(cube0.data)
# Loop over velocity slices.
for i in range(cube0.data.shape[0]):
    # Apply a 90-degree counterclockwise rotation to slice i.
    rotated_data[i] = np.flip(cube0.data[i],axis=0)#np.flip(np.rot90(merged_data[0][i], k=0),axis=0)
    #rotated_data[i] = np.flip(np.rot90(cube0.data[0][i], k=3),axis=0)

#merged_data[0] = np.rot90(merged_data[0],k=2)
merged_data_0 = np.nansum(rotated_data, axis=0)
plt.imshow(merged_data_0, origin='lower')
plt.colorbar()
plt.savefig(os.path.join(output_folder_path, 'Cube_'+str(object_id)+'_0.png'))


In [ ]:

polaris_object_header

In [ ]:
ft = SpectralCube.read('./Cube_56t2_0.fits')
#ft = SpectralCube.read(output_folder_path+'/Cube_810_0.fits')

ftm0 = ft.moment0().hdu 

plt.imshow(ftm0.data, origin='lower',extent=[0,1,-0.5,0.5])
plt.xlabel('Y (kpc)')
plt.ylabel('Z (kpc)')
plt.colorbar()

In [ ]:
ftm0.data.shape

In [ ]:
# Example usage.
folder_path = polaris_object_dir  # Replace with your directory path.
output_path = polaris_object_dir+'/output.fits'  # Output file path.
process_folder(folder_path, output_path)

In [ ]:
from spectral_cube import SpectralCube

In [ ]:
f57 = SpectralCube.read('/Volumes/MWISP/work/polaris_object/57/Cube_57_0.fits')
f68 = SpectralCube.read('/Volumes/MWISP/work/polaris_object/68_150km/Cube_68_150km_0.fits')
f57

In [ ]:
lv57 = f57.moment(order=0,axis=1)
lv68 = f68.moment(order=0,axis=1)
plt.imshow(lv57.hdu.data, origin='lower',extent=[5,7,0-75,150-75],aspect='auto',vmax=100,vmin=10)
plt.imshow(lv68.hdu.data, origin='lower',extent=[6,8,0+7-75,150+7-75],aspect='auto',vmax=100,vmin=10)
plt.xlim(5,8)

In [ ]:
# Use subplots to display two plots together.
plt.figure(figsize=(6, 12))  # Set the overall figure size.

# First subplot.
plt.subplot(2, 1, 1)  # (rows, columns, subplot index)
plt.imshow(lv57.hdu.data, origin='lower',extent=[5,7,0,150],aspect='auto')

plt.xlabel('Galactic Height (kpc)')
plt.ylabel('Velocity (km/s)')


# Second subplot.
plt.subplot(2, 1, 2)  # Change the subplot index to create another panel.
plt.imshow(lv68.hdu.data, origin='lower',extent=[6,8,-75,75],aspect='auto')

plt.xlabel('Galactic Height (kpc)')
plt.ylabel('Velocity (km/s)')

